In [1]:
import pandas as pd
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path.cwd().resolve()
if NOTEBOOKS_DIR.name != "notebooks":
    if (NOTEBOOKS_DIR / "notebooks").exists():
        NOTEBOOKS_DIR = NOTEBOOKS_DIR / "notebooks"
    elif NOTEBOOKS_DIR.parent.name == "notebooks":
        NOTEBOOKS_DIR = NOTEBOOKS_DIR.parent
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

from config import PATHS

gt_df = pd.read_csv(PATHS["explanation_groundtruth_sheet1_csv"], header=5)
gt_df = gt_df.iloc[:, [0, 1, 3, 7, 9]]
gt_df.columns = ['geneA', 'geneB', 'explanation', 'label_true', 'important_features']
gt_df = gt_df[gt_df['label_true'] == "1"]  # Filter rows where label_true is 1
gt_df.head()

,geneA,geneB,explanation,label_true,important_features
1,KRAS,SNRPD3,KRAS and SNRPD3 have a synthetic lethality rel...,1,spliceosome complex\nalternative splicing\ncel...
6,EGFR,MYC,EGFR and MYC have synthetic lethality relation...,1,survival\nregulating cell proliferation\n
8,EGFR,MET,EGFR and MET have a synthetic lethality relati...,1,induces apoptosis\nreduces cell proliferation
9,AKT1,SMAD4,AKT1 and SMAD4 have a synthetic lethality rela...,1,mitotic spindle assembly\ncell death
10,PTEN,RNF126,"According to this literature, PTEN and RNF126 ...",1,cell proliferation\nmetastasis\ntumor formation


In [2]:
gt_df2 = pd.read_csv(PATHS["explanation_groundtruth_pubmed_csv"], header=None)
gt_df2 = gt_df2.iloc[:, [0, 1, 4]]
gt_df2.columns = ['geneA', 'geneB', 'explanation']
gt_df2['label_true'] = 1
gt_df2['important_features'] = 'spaceholder'
gt_df2.head()

,geneA,geneB,explanation,label_true,important_features
0,TP53,LDHA,TP53 and LDHA have a synthetic lethality relat...,1,spaceholder
1,BCL2,COPS5,BCL2 and COPS5 have a synthetic lethality rela...,1,spaceholder
2,FGFR3,ITGB3,FGFR3 and ITGB3 have a synthetic lethality rel...,1,spaceholder
3,KIT,DAG1,KIT and DAG1 have a synthetic lethality relati...,1,spaceholder
4,CDKN1A,SUMO1,CDKN1A and SUMO1 have a synthetic lethality re...,1,spaceholder


In [3]:
gt_df = pd.concat([gt_df, gt_df2], ignore_index=True)
gt_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   geneA               138 non-null    object
 1   geneB               138 non-null    object
 2   explanation         138 non-null    object
 3   label_true          138 non-null    object
 4   important_features  133 non-null    object
dtypes: object(5)
memory usage: 5.5+ KB


In [4]:
# Import necessary libraries and define required functions
import os
import pickle as pkl
import numpy as np

# Load embeddings
def load_slformer_embeddings(embedding_type='cross', model_variant='kg', base_path=None):
    """Load SLformer cross-attention or transformer embeddings."""
    if base_path is None:
        base_path = PATHS["local_all_sl_dir"]
    
    if embedding_type == 'cross':
        filename = f"mix_slformer_{model_variant}_crossemb.pkl"
    elif embedding_type == 'transformer':
        filename = f"mix_slformer_{model_variant}_transformeremb.pkl"
    else:
        raise ValueError("embedding_type must be 'cross' or 'transformer'")
    
    filepath = os.path.join(base_path, filename)
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Embedding file not found: {filepath}")
    
    with open(filepath, 'rb') as f:
        embeddings = pkl.load(f)
    
    return embeddings

# Define the gene pair search function
def get_embedding_for_gene_pair(embeddings, gene1, gene2, cancer, pred_df, common_data=None):
    """Extract embeddings for a specific gene pair and cancer type across all CV folds."""
    results = []
    
    # Create mask for gene pair (both orientations)
    if cancer:
        mask = (((pred_df['primary_gene'] == gene1) & (pred_df['partner_gene'] == gene2)) |
                ((pred_df['primary_gene'] == gene2) & (pred_df['partner_gene'] == gene1))) & \
               (pred_df['cancer'] == cancer)
    else:
        mask = ((pred_df['primary_gene'] == gene1) & (pred_df['partner_gene'] == gene2)) | \
               ((pred_df['primary_gene'] == gene2) & (pred_df['partner_gene'] == gene1))
    
    matching_rows = pred_df[mask]
    
    if len(matching_rows) == 0:
        return pd.DataFrame()
    
    # Group by CV fold
    fold_size = len(pred_df) // len(embeddings) if len(embeddings) > 0 else len(pred_df)
    
    for _, row in matching_rows.iterrows():
        original_idx = row.name
        fold_idx = original_idx // fold_size
        fold_idx = min(fold_idx, len(embeddings) - 1)
        local_idx = original_idx % fold_size
        
        if fold_idx < len(embeddings):
            fold_embeddings = embeddings[fold_idx]
            
            # Extract embeddings for this instance
            embedding_data = {}
            for i, emb_part in enumerate(fold_embeddings):
                if local_idx < emb_part.shape[0]:
                    embedding_data[f'embedding_part_{i}'] = emb_part[local_idx]
            
            # Create result record
            result_record = {
                'context': row['cancer'],
                'SLscore': row['score'], 
                'Label': row['label'],
                'fold': fold_idx,
                'primary_gene': row['primary_gene'],
                'partner_gene': row['partner_gene']
            }
            
            # Add embedding parts
            result_record.update(embedding_data)
            results.append(result_record)
    
    if results:
        result_df = pd.DataFrame(results)
        result_df = result_df.rename(columns={
            'embedding_part_0': 'primary_embedding', 
            'embedding_part_1': 'partner_embedding'
        })
        return result_df
    else:
        return pd.DataFrame()

cross_emb_kg = load_slformer_embeddings(embedding_type='cross', model_variant='kg')

gene_preds = pd.DataFrame()
for i in range(5):
    path = os.path.join(PATHS["local_all_sl_dir"], f'pred_mix_slformer_kg_cv{i+1}.csv')
    truth_path = PATHS["benchmark_mix_all_test_fold_template"].format(fold=i + 1)
    df = pd.read_csv(path)
    true_df = pd.read_csv(truth_path)
    df['label'] = true_df['label']
    gene_preds = pd.concat([gene_preds, df], ignore_index=True)
    
print(f"Loaded {len(cross_emb_kg)} embedding folds and {gene_preds.shape[0]} gene predictions")

Loaded 5 embedding folds and 58620 gene predictions


In [5]:
# Search for all gene pairs from ground truth data
search_results = []

print(f"Searching for {len(gt_df)} gene pairs from ground truth data...")

no_item_count = 0
for idx, row in gt_df.iterrows():
    gene1 = row['geneA']
    gene2 = row['geneB']
    result_df = get_embedding_for_gene_pair(
        embeddings=cross_emb_kg,
        gene1=gene1,
        gene2=gene2, 
        cancer=None,
        pred_df=gene_preds
    )
    
    if result_df.empty:
        no_item_count += 1
        continue
    if result_df[result_df['Label'] == 1].empty:
        print(f'end= {gene1}-{gene2} has no items with Label == 1')
        continue
    item_count = len(result_df)
    search_results.append({
        'geneA': gene1,
        'geneB': gene2,
        'item_count': item_count,
        'explanation': row['explanation'],
        'important_features': row['important_features'],
        'result_df': result_df
    })
    
    
    
print(f"\nCompleted search for {len(search_results)} gene pairs.")

# Convert to DataFrame and sort by item count in descending order
results_summary = pd.DataFrame([{
    'geneA': r['geneA'],
    'geneB': r['geneB'], 
    'item_count': r['item_count'],
    'explanation': r['explanation'],
    'important_features': r['important_features']
} for r in search_results])

results_summary = results_summary.sort_values('item_count', ascending=False)

print(f"\nResults sorted by item count (descending):")
print(results_summary.to_string(index=False))

# Also create a dictionary for easy access to the detailed results
results_dict = {f"{r['geneA']}-{r['geneB']}": r['result_df'] for r in search_results}

print(f"\nSummary:")
print(f"Total gene pairs searched: {len(results_summary)}")
print(f"Gene pairs with results: {len(results_summary[results_summary['item_count'] > 0])}")
print(f"Gene pairs with no results: {no_item_count}")

Searching for 138 gene pairs from ground truth data...
end= AKT1-SMAD4 has no items with Label == 1
end= AKT1-CDK6 has no items with Label == 1
end= CDK12-PARP1 has no items with Label == 1
end= ERBB2-PIK3CA has no items with Label == 1
end= AKT1-BCL2 has no items with Label == 1
end= BCL2-CDK6 has no items with Label == 1
end= AKT1-MTOR has no items with Label == 1
end= BRCA2-MYC has no items with Label == 1
end= BRCA1-CHEK2 has no items with Label == 1
end= BRCA2-CHEK1 has no items with Label == 1
end= CDKN1A-KRAS has no items with Label == 1
end= ATM-MYC has no items with Label == 1
end= ATM-PTEN has no items with Label == 1
end= SMARCA4-CDK4 has no items with Label == 1
end= VHL-MET has no items with Label == 1
end= VHL-MTOR has no items with Label == 1
end= ATM-ATR has no items with Label == 1
end= PTEN-BRCA2 has no items with Label == 1
end= PIM1-MYC has no items with Label == 1
end= KRAS-CDKN1A has no items with Label == 1
end= TP53-CHEK1 has no items with Label == 1
end= PTEN-A

In [6]:
# Display the results summary
print("=== SEARCH RESULTS SUMMARY ===")
print(f"Total gene pairs searched: {len(results_summary)}")
print(f"Gene pairs with results: {len(results_summary[results_summary['item_count'] > 0])}")
print(f"Gene pairs with no results: {len(results_summary[results_summary['item_count'] == 0])}")

print("\n=== ALL RESULTS (sorted by item count, descending) ===")
results_summary

=== SEARCH RESULTS SUMMARY ===
Total gene pairs searched: 10
Gene pairs with results: 10
Gene pairs with no results: 0

=== ALL RESULTS (sorted by item count, descending) ===


,geneA,geneB,item_count,explanation,important_features
6,KRAS,PSMB5,2,KRAS and PSMB5 have a synthetic lethality rela...,"proteasome inhibition, accumulation of p53"
4,ARID1A,ALDH2,2,ARID1A and ALDH2 have a synthetic lethality re...,accumulation of reactive oxygen species (ROS) ...
1,BRCA2,EZH2,1,BRCA2 and EZH2 have a synthetic lethality rela...,DNA damage response \nchromatin remodeling
0,PTEN,RNF126,1,"According to this literature, PTEN and RNF126 ...",cell proliferation\nmetastasis\ntumor formation
3,KRAS,ERH,1,KRAS and ERH have a synthetic lethality relati...,mRNA splicing and mitosis
2,KRAS,PLK1,1,KRAS and PLK1 have a synthetic lethality relat...,mitotic defects and apoptosis
5,KRAS,GATA2,1,"According to one study[^3^], KRAS and GATA2 ha...","proteasome production, DNA damage response"
7,KRAS,THOC1,1,KRAS and THOC1 have a synthetic lethality rela...,required for proliferation
8,TRRAP,MYC,1,TRRAP and MYC have a synthetic lethality relat...,NaN
9,ESR1,KRAS,1,"According to the existing literature[^1^], ESR...",spaceholder


In [7]:
results_summary

,geneA,geneB,item_count,explanation,important_features
6,KRAS,PSMB5,2,KRAS and PSMB5 have a synthetic lethality rela...,"proteasome inhibition, accumulation of p53"
4,ARID1A,ALDH2,2,ARID1A and ALDH2 have a synthetic lethality re...,accumulation of reactive oxygen species (ROS) ...
1,BRCA2,EZH2,1,BRCA2 and EZH2 have a synthetic lethality rela...,DNA damage response \nchromatin remodeling
0,PTEN,RNF126,1,"According to this literature, PTEN and RNF126 ...",cell proliferation\nmetastasis\ntumor formation
3,KRAS,ERH,1,KRAS and ERH have a synthetic lethality relati...,mRNA splicing and mitosis
2,KRAS,PLK1,1,KRAS and PLK1 have a synthetic lethality relat...,mitotic defects and apoptosis
5,KRAS,GATA2,1,"According to one study[^3^], KRAS and GATA2 ha...","proteasome production, DNA damage response"
7,KRAS,THOC1,1,KRAS and THOC1 have a synthetic lethality rela...,required for proliferation
8,TRRAP,MYC,1,TRRAP and MYC have a synthetic lethality relat...,NaN
9,ESR1,KRAS,1,"According to the existing literature[^1^], ESR...",spaceholder


In [8]:
for index, item in results_summary.iterrows():
    explanation = item['explanation']
    geneA, geneB = item['geneA'], item['geneB']
    pair_name = f'{geneA}-{geneB}'
    output_dir = os.path.join(PATHS["local_explanation_groundtruth_dir"], pair_name)
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, 'explanation.txt'), 'w') as f:
        f.buffer.write(explanation.encode('utf-8'))
        f.close()
results_summary[['geneA', 'geneB', 'important_features']].to_csv(
    os.path.join(PATHS["local_explanation_groundtruth_dir"], 'important_features.csv')
)

In [9]:
for idx, row in results_summary.iterrows():
    gene1 = row['geneA']
    gene2 = row['geneB']
    result_df = get_embedding_for_gene_pair(
                embeddings=cross_emb_kg,
                gene1=gene1,
                gene2=gene2, 
                cancer=None,
                pred_df=gene_preds
            )
    print(result_df[['primary_gene', 'partner_gene', 'SLscore', 'Label', 'context']])
    print('=' * 50)

  primary_gene partner_gene   SLscore  Label context
0         KRAS        PSMB5  0.004934      1    COAD
1         KRAS        PSMB5  0.896639      1    LUAD
  primary_gene partner_gene   SLscore  Label context
0        ALDH2       ARID1A  0.009400      1    CESC
1        ALDH2       ARID1A  0.032504      1    LUAD
  primary_gene partner_gene   SLscore  Label context
0        BRCA2         EZH2  0.922231      1    BRCA
  primary_gene partner_gene   SLscore  Label context
0         PTEN       RNF126  0.004276      1    COAD
  primary_gene partner_gene   SLscore  Label context
0          ERH         KRAS  0.852795      1    LUAD
  primary_gene partner_gene   SLscore  Label context
0         KRAS         PLK1  0.943691      1    LUAD
  primary_gene partner_gene   SLscore  Label context
0        GATA2         KRAS  0.852047      1    LUAD
  primary_gene partner_gene   SLscore  Label context
0         KRAS        THOC1  0.564279      1    LUAD
  primary_gene partner_gene   SLscore  Label c

In [10]:
result_df = get_embedding_for_gene_pair(
                embeddings=cross_emb_kg,
                gene1='TP53',
                gene2='IGF2', 
                cancer=None,
                pred_df=gene_preds
            )
print(result_df[['primary_gene', 'partner_gene', 'SLscore', 'Label', 'context']] if not result_df.empty else print("No results found for the pair"))

No results found for the pair
None
